In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("cleaned_ebay_deals.csv")

df.head()

dataset preview

In [ ]:
print(df.shape)
print(df.isnull().sum())

In [ ]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['shipping'] = pd.to_numeric(df['shipping'], errors='coerce')
df['original_price'] = pd.to_numeric(df['original_price'], errors='coerce')

In [ ]:
df['effective_price'] = df['price'] + df['shipping']

df['discount_ratio'] = (
    (df['original_price'] - df['price']) / df['original_price']
)

In [ ]:
df['title_len'] = df['title'].str.len()
df['title_word_count'] = df['title'].str.split().str.len()

df['has_new'] = df['title'].str.contains('new', case=False, na=False).astype(int)
df['has_used'] = df['title'].str.contains('used', case=False, na=False).astype(int)
df['has_refurbished'] = df['title'].str.contains('refurbished', case=False, na=False).astype(int)
df['has_bundle'] = df['title'].str.contains('bundle', case=False, na=False).astype(int)
df['has_case'] = df['title'].str.contains('case', case=False, na=False).astype(int)
df['has_charger'] = df['title'].str.contains('charger', case=False, na=False).astype(int)

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour'] = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.weekday

engineered dataset

In [ ]:
df.to_csv("ebay_features.csv", index=False)

In [ ]:
df['high_discount'] = df['discount_percentage'] > 20
baseline_features = ['price', 'original_price']
extended_features = [
    'price', 'original_price', 'effective_price', 'discount_ratio',
    'title_len', 'title_word_count',
    'has_new', 'has_used', 'has_refurbished',
    'has_bundle', 'has_case', 'has_charger',
    'hour', 'weekday'
]

In [1]:
from sklearn.model_selection import train_test_split

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
X = df[extended_features]
y = df['high_discount']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split (..., test_size=0.2, random_state=42, stratify=y)

In [ ]:
from sklearn.linear_model import LogisticRegression

model1 = LogisticRegression()
model1.fit(df[baseline_features], y)

model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train, y_train)

from sklearn.tree import DecisionTreeClassifier

model3 = DecisionTreeClassifier()
model3.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
print(df['high_discount'].value_counts(normalize=True))

# sampling

majority = df[df['high_discount'] == False]
minority = df[df['high_discount'] == True]

balanced = pd.concat([
    majority.sample(len(minority), random_state=42),
    minority
])

In [ ]:
X_bal = balanced[extended_features]
y_bal = balanced['high_discount']